In [1]:
import pandas as pd
import rasterio
import numpy as np

In [2]:
eventos = pd.read_csv('../data/cafe/eventos_zona_cafetera.csv')
no_eventos = pd.read_csv('../data/cafe/noeventos_zona_cafetera.csv')

In [3]:
len(eventos), len(no_eventos)

(2163, 2163)

In [12]:
no_eventos.columns

Index(['id', 'fecha_evento', 'X', 'Y', 'Mes', 'rain_3d', 'rain_7d', 'rain_15d',
       'rain_30d', 'heavy_rain_days', 'max_daily_rain', 'soil_moisture_proxy',
       'antecedent_moisture_x_maxrain', 'precip_temp_ratio', 'humidity_index',
       'mean_precip', 'mean_temp', 'mean_humidity', 'temp_range',
       'precip_variability', 'dry_days_before', 'slope_x_rain3d', 'mean_et0',
       'total_et0', 'water_balance', 'lon', 'lat', 'slope', 'altura',
       'target'],
      dtype='object')

In [11]:
#eliminar columnas innecesarias
no_eventos = no_eventos.drop(columns=['Unnamed: 0'])

In [33]:
eventos["target"] = 1
no_eventos["target"] = 0

In [37]:
no_eventos.rename(columns={
    "slope1": "slope",
    "altura1": "altura",}, 
               inplace=True)

In [44]:
no_eventos['slope_x_rain3d'] = no_eventos['slope'] * no_eventos['rain_3d']

In [15]:
no_eventos.head()

,id,fecha_evento,X,Y,Mes,rain_3d,rain_7d,rain_15d,rain_30d,heavy_rain_days,...,dry_days_before,slope_x_rain3d,mean_et0,total_et0,water_balance,lon,lat,slope,altura,target
0,01,2013/04/23,-74.519659,3.611665,4,64.1,130.6,174.0,256.9,1,...,0,309.124954,2.885667,86.57,170.33,4.831281e+06,1.957234e+06,4.822542,3608,0
1,02,2006/07/31,-75.822492,2.119139,7,19.4,30.8,51.4,183.7,1,...,0,148.750999,2.514667,75.44,108.26,4.686139e+06,1.792471e+06,7.667577,857,0
2,03,2004/07/27,-74.784400,3.820276,7,5.2,8.7,35.1,82.7,0,...,0,100.044468,3.875667,116.27,-33.57,4.801927e+06,1.980348e+06,19.239321,582,0
3,04,2002/05/22,-75.468876,2.454582,5,7.3,15.3,32.9,180.4,1,...,0,271.948169,2.674333,80.23,100.17,4.725551e+06,1.829501e+06,37.253174,1262,0
4,05,2006/08/05,-74.954292,5.981556,8,0.7,5.9,71.4,111.2,0,...,0,26.435118,4.808000,144.24,-33.04,4.783758e+06,2.219335e+06,37.764454,688,0


In [17]:
# Unir ambos en un solo DataFrame
df = pd.concat([eventos,no_eventos], ignore_index=True)

In [18]:
  
#ver cuantos casos positivos y negativos
print(f"Total combinado: {len(df)}")
print(df['target'].value_counts())

Total combinado: 4326
target
1    2163
0    2163
Name: count, dtype: int64


In [61]:
df.columns

Index(['id', 'fecha_evento', 'X', 'Y', 'Mes', 'rain_3d', 'rain_7d', 'rain_15d',
       'rain_30d', 'heavy_rain_days', 'max_daily_rain', 'soil_moisture_proxy',
       'antecedent_moisture_x_maxrain', 'precip_temp_ratio', 'humidity_index',
       'mean_precip', 'mean_temp', 'mean_humidity', 'temp_range',
       'precip_variability', 'dry_days_before', 'slope_x_rain3d', 'mean_et0',
       'total_et0', 'water_balance', 'lon', 'lat', 'slope', 'altura',
       'target'],
      dtype='object')

In [19]:
import sys
import os

# Añade la ruta donde está utils.py
sys.path.append(os.path.abspath("../app"))

from model import train_siatma_model

In [20]:
# Entrenar el modelo
modelo, datos_test = train_siatma_model(df)


Iniciando entrenamiento del Sistema SIATMA
Dataset: 4326 registros, 30 columnas
Distribución target: {1: 2163, 0: 2163}
=== INICIANDO ENTRENAMIENTO DEL ENSEMBLE SIATMA ===

Advertencia: Se encontraron valores faltantes. Rellenando con mediana...
Tamaño del conjunto de entrenamiento: 2595
Tamaño del conjunto de validación: 865
Tamaño del conjunto de prueba: 866

Entrenando Random Forest...
Top 10 características más importantes (Random Forest):
          feature  importance
26         altura    0.094542
14      mean_temp    0.088282
0               X    0.085708
23            lon    0.082159
1               Y    0.072083
24            lat    0.070921
25          slope    0.046800
20       mean_et0    0.036345
2             Mes    0.035872
15  mean_humidity    0.034812
Entrenando XGBoost...
Top 10 características más importantes (XGBoost):
        feature  importance
23          lon    0.070033
26       altura    0.065373
14    mean_temp    0.063926
24          lat    0.055208
1         

Modelo guardado con prefijo: siatma_ensemble_v1


In [21]:
modelo.feature_columns

['X',
 'Y',
 'Mes',
 'rain_3d',
 'rain_7d',
 'rain_15d',
 'rain_30d',
 'heavy_rain_days',
 'max_daily_rain',
 'soil_moisture_proxy',
 'antecedent_moisture_x_maxrain',
 'precip_temp_ratio',
 'humidity_index',
 'mean_precip',
 'mean_temp',
 'mean_humidity',
 'temp_range',
 'precip_variability',
 'dry_days_before',
 'slope_x_rain3d',
 'mean_et0',
 'total_et0',
 'water_balance',
 'lon',
 'lat',
 'slope',
 'altura']